In [1]:
pip install libpysal

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 11.1 MB/s eta 0:00:00
Using cached scikit_learn-1.9.0-cp312-cp312-win_amd64.whl (8.2 MB)
Using cached scipy-1.18.0-cp312-cp312-win_amd64.whl (36.6 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl (15 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Not

In [2]:
pip install esda

  Using cached esda-2.10.0-py3-none-any.whl.metadata (2.4 kB)
Using cached esda-2.10.0-py3-none-any.whl (247 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import geopandas as gpd
import pandas as pd
import libpysal
from libpysal.weights import Queen, higher_order
from esda.moran import Moran
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import KNN

In [ ]:
# setting wd 
import os
#os.chdir('/users/sarazarea26/fooddesertproject')

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/users/sarazarea26/fooddesertproject'

In [8]:
import geopandas as gpd
import numpy as np

# Load GeoPackage
dallas = gpd.read_file("dallas_modeling_final.gpkg")
tarrant = gpd.read_file("tarrant_modeling_final.gpkg")
travis = gpd.read_file("travis_modeling_final.gpkg")
bexar = gpd.read_file("bexar_modeling_final.gpkg")
harris = gpd.read_file("harris_modeling_final.gpkg")

In [10]:
# Reproject all counties to the same CRS
target_crs = "EPSG:3083"

dallas = dallas.to_crs(target_crs)
tarrant = tarrant.to_crs(target_crs)
travis = travis.to_crs(target_crs)
bexar = bexar.to_crs(target_crs)
harris = harris.to_crs(target_crs)

In [11]:
#merge 
dallas["county"] = "Dallas"
tarrant["county"] = "Tarrant"
travis["county"] = "Travis"
bexar["county"] = "Bexar"
harris["county"] = "Harris"

all_counties = pd.concat(
    [dallas, tarrant, travis, bexar, harris],
    ignore_index=True
)

In [12]:
all_counties.to_csv("all_counties.csv", index=False)

In [5]:
county = "dallas"

In [ ]:
gdf = gpd.read_file(f"modeling_data/{county}_modeling_final.gpkg")
print(gdf.shape)  
gdf.columns

DataSourceError: modeling_data/dallas_modeling_final.gpkg: No such file or directory

In [ ]:
# gdf = gdf.dropna(subset=["median_income"]).copy()
# print(gdf_income.shape)  

(1271, 24)


In [ ]:
# only use when calculating k, otherwise comment out 
n = len(gdf)
k = round(n ** (1/3))
print(f"Number of census tracts: {n}")
print(f"Cube root of n: {n**(1/3):.2f}")
print(f"Selected k: {k}")

Number of census tracts: 669
Cube root of n: 8.75
Selected k: 9


In [ ]:
# build first-order contiguity weights
# w1 = Queen.from_dataframe(gdf, use_index=True)

w1 = KNN.from_dataframe(gdf, k=k, use_index=True)
print(w1.n)

669


In [ ]:
def run_correlogram(var_name, gdf, w1, max_lag=6):
    var = gdf[var_name].values
    results = []
    for lag in range(1, max_lag + 1):
        w_lag = higher_order(w1, lag)
        w_lag.transform = "R"
        mi = Moran(var, w_lag, permutations=999)
        results.append({
            "variable": var_name,
            "lag": lag,
            "I": mi.I,
            "expected_I": mi.EI,
            "se_I": mi.seI_sim,
            "p_sim": mi.p_sim
        })
    return pd.DataFrame(results)


# run the three variables
diabetes_df = run_correlogram("E_DIABETES", gdf, w1)
chd_df = run_correlogram("E_CHD", gdf, w1)
walking_df = run_correlogram("walking_ind", gdf, w1)
poverty_df = run_correlogram("below_200_fed_poverty_percentage", gdf, w1)
uninsured_df = run_correlogram("uninsured", gdf, w1)
uf_df = run_correlogram("distance_to_nearest_uf", gdf, w1)
# income_df = run_correlogram("median_income", gdf, w1)
grocery_df = run_correlogram("distance_to_nearest_grocery", gdf, w1)
# mobility_df= run_correlogram("pct_no_vehicle", gdf, w1)
fm_df= run_correlogram("distance_to_nearest_fm", gdf, w1)


# combine and save the table
all_results = pd.concat([diabetes_df, chd_df, walking_df, poverty_df, uninsured_df, uf_df, grocery_df, fm_df], ignore_index=True)
all_results.to_csv(f"{county}_correlogram_results_KNN.csv", index=False)
all_results

,variable,lag,I,expected_I,se_I,p_sim
0,E_DIABETES,1,0.284086,-0.001497,0.015631,0.001
1,E_DIABETES,2,-0.001578,-0.001497,0.011234,0.387
2,E_DIABETES,3,-0.001989,-0.001497,0.009138,0.452
3,E_DIABETES,4,-0.003526,-0.001497,0.008086,0.468
4,E_DIABETES,5,-0.006572,-0.001497,0.006673,0.251
5,E_DIABETES,6,-0.008051,-0.001497,0.006004,0.122
6,E_CHD,1,0.283542,-0.001497,0.016447,0.001
7,E_CHD,2,-0.003605,-0.001497,0.011565,0.440
8,E_CHD,3,-0.005104,-0.001497,0.009399,0.419
9,E_CHD,4,-0.007185,-0.001497,0.007618,0.264


In [ ]:
# now plot and save
for var_name, df in [
    ("E_DIABETES", diabetes_df),
    ("E_CHD", chd_df),
    ("walking_ind", walking_df),
    ("below_200_fed_poverty_percentage", poverty_df),
    ("uninsured", uninsured_df),
    ("distance_to_nearest_uf", uf_df),
    ("distance_to_nearest_grocery", grocery_df),
    ("distance_to_nearest_fm", fm_df)
]:
    plt.figure(figsize=(7, 4))

    plt.errorbar(
        df["lag"],
        df["I"],
        yerr=2 * df["se_I"],
        marker="o",
        color="steelblue",
        ecolor="steelblue",
        elinewidth=1,
        capsize=4
    )

    plt.axhline(0, color="gray", linestyle="--", alpha=0.6)
    plt.xlabel("Lag order")
    plt.ylabel("Moran's I")
    plt.title(f"Spatial Correlogram — {var_name}")

    plt.savefig(
        f"{county}_correlogram_{var_name}_KNN.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()